In [5]:
import sys
sys.path.append("../") # Go up one level to project root 
#from src_test import drugbank_parse_fast
import pandas as pd
import zipfile
import xml.etree.ElementTree as ET
import random
import copy
import os
from collections import Counter

In [6]:
NS = "{http://www.drugbank.ca}"

drug_tag = f"{NS}drug"

In [9]:
records = []

total_drugs = 0
drugs_seen = 0

with zipfile.ZipFile(
    "../data/raw/drugbank_full_database_V5.1.14.zip", "r"
) as z:

    xml_filename = [f for f in z.namelist() if f.endswith(".xml")][0]
    print("Streaming:", xml_filename)

    with z.open(xml_filename) as xml_file:

        for event, elem in ET.iterparse(xml_file, events=("end",)):

            # only process full drug nodes
            if elem.tag != drug_tag:
                continue

            total_drugs += 1

            if total_drugs % 1000 == 0:
                print(f"Inspected {total_drugs} drugs | Extracted {drugs_seen}")

            # -----------------------------
            # PRIMARY DRUGBANK ID
            # -----------------------------
            primary_id = None
            for d in elem.findall(f"{NS}drugbank-id"):
                if d.get("primary") == "true":
                    primary_id = d.text
                    break

            # -----------------------------
            # DRUG NAME
            # -----------------------------
            name_elem = elem.find(f"{NS}name")
            drug_name = name_elem.text if name_elem is not None else None

            # -----------------------------
            # FILTER: small molecule + approved + not withdrawn
            # -----------------------------
            groups = [
                g.text for g in elem.findall(f"{NS}groups/{NS}group")
                if g.text
            ]

            if not (
                elem.get("type") == "small molecule"
                and primary_id is not None
                and "approved" in groups
                and "withdrawn" not in groups
            ):
                elem.clear()
                continue

            # =====================================================
            # ATC CODES
            # =====================================================
            atc_codes = [
                a.get("code")
                for a in elem.findall(f"{NS}atc-codes/{NS}atc-code")
                if a.get("code")
            ]

            # =====================================================
            # DRUG INTERACTIONS
            # =====================================================
            interactions = []

            di_block = elem.find(f"{NS}drug-interactions")

            if di_block is not None:

                for di in di_block.findall(f"{NS}drug-interaction"):

                    i_id = di.find(f"{NS}drugbank-id")
                    i_name = di.find(f"{NS}name")
                    i_desc = di.find(f"{NS}description")

                    interactions.append([
                        i_id.text if i_id is not None else None,
                        i_name.text if i_name is not None else None,
                        i_desc.text if i_desc is not None else None
                    ])

            # -----------------------------
            # STORE RECORD
            # -----------------------------
            records.append({
                "drugbank_id": primary_id,
                "drug_name": drug_name,
                "ATC": atc_codes,
                "drug_interactions": interactions
            })

            drugs_seen += 1

            print(f"[MATCH {drugs_seen}] {primary_id} | {drug_name}")

            elem.clear()

print("\n================ SUMMARY ================")
print("Total drugs inspected:", total_drugs)
print("Filtered drugs extracted:", drugs_seen)

df = pd.DataFrame(records)
print(df.shape)
df.head()

Streaming: drugbank_full_database_V5.1.14.xml
[MATCH 1] DB00006 | Bivalirudin
[MATCH 2] DB00014 | Goserelin
[MATCH 3] DB00027 | Gramicidin D
[MATCH 4] DB00035 | Desmopressin
[MATCH 5] DB00050 | Cetrorelix
[MATCH 6] DB00080 | Daptomycin
[MATCH 7] DB00091 | Cyclosporine
Inspected 1000 drugs | Extracted 7
Inspected 2000 drugs | Extracted 7
[MATCH 8] DB00114 | Pyridoxal phosphate
[MATCH 9] DB00115 | Cyanocobalamin
Inspected 3000 drugs | Extracted 9
Inspected 4000 drugs | Extracted 9
[MATCH 10] DB00117 | Histidine
Inspected 5000 drugs | Extracted 10
Inspected 6000 drugs | Extracted 10
Inspected 7000 drugs | Extracted 10
Inspected 8000 drugs | Extracted 10
Inspected 9000 drugs | Extracted 10
Inspected 10000 drugs | Extracted 10
Inspected 11000 drugs | Extracted 10
Inspected 12000 drugs | Extracted 10
[MATCH 11] DB00118 | Ademetionine
Inspected 13000 drugs | Extracted 11
[MATCH 12] DB00119 | Pyruvic acid
[MATCH 13] DB00120 | Phenylalanine
Inspected 14000 drugs | Extracted 13
[MATCH 14] DB0012

,drugbank_id,drug_name,ATC,drug_interactions
0,DB00006,Bivalirudin,[B01AE06],"[[DB06605, Apixaban, Apixaban may increase the..."
1,DB00014,Goserelin,[L02AE03],"[[DB09066, Corifollitropin alfa, The therapeut..."
2,DB00027,Gramicidin D,[R02AB30],"[[DB12768, BCG vaccine, The therapeutic effica..."
3,DB00035,Desmopressin,[H01BA02],"[[DB00564, Carbamazepine, The risk or severity..."
4,DB00050,Cetrorelix,[H01CC02],[]


In [10]:
df[df["drug_name"].str.lower() == "bivalirudin"]["drug_interactions"].iloc[0]

[['DB06605',
  'Apixaban',
  'Apixaban may increase the anticoagulant activities of Bivalirudin.'],
 ['DB06695',
  'Dabigatran etexilate',
  'Dabigatran etexilate may increase the anticoagulant activities of Bivalirudin.'],
 ['DB01254',
  'Dasatinib',
  'The risk or severity of bleeding and hemorrhage can be increased when Dasatinib is combined with Bivalirudin.'],
 ['DB01609',
  'Deferasirox',
  'The risk or severity of gastrointestinal bleeding can be increased when Bivalirudin is combined with Deferasirox.'],
 ['DB01586',
  'Ursodeoxycholic acid',
  'The risk or severity of bleeding and bruising can be increased when Bivalirudin is combined with Ursodeoxycholic acid.'],
 ['DB02123',
  'Glycochenodeoxycholic Acid',
  'The risk or severity of bleeding and bruising can be increased when Bivalirudin is combined with Glycochenodeoxycholic Acid.'],
 ['DB02659',
  'Cholic Acid',
  'The risk or severity of bleeding and bruising can be increased when Bivalirudin is combined with Cholic Acid.

In [11]:
output_path = "../data/raw/drugbank_approved_small_2369_drugs.csv"
df.to_csv(output_path, index=False)

In [16]:
def print_tree(elem, ns, indent=0, out=None):
    """recursively print the nested tree elements"""
    local_tag = elem.tag.replace(f"{{{ns}}}","")
    attribs = dict(elem.attrib)
    text = (elem.text or "").strip()
    if len(text) > 80:
        text = text[:80] + "..."

    attrib_str = f"{attribs}" if attribs else ""
    text_str = f" {text}" if text else ""
    line = f"{'  ' * indent}{local_tag}{attrib_str}{text_str}\n"  

    if out: 
        out.write(line)
    else:
        print(line, end="") # fallback to console if no file given

    for child in elem:
        print_tree(child, ns, indent +1, out=out)

In [18]:
drugs_seen = 0
with open("../data/raw/drugbank_inspection.txt", "w") as out:
    with zipfile.ZipFile("../data/raw/drugbank_full_database_V5.1.14.zip", 'r') as z:
        xml_filename = [f for f in z.namelist() if f.endswith('.xml')][0]
        print(f"Streaming: {xml_filename}")

        with z.open(xml_filename)as xml_file:
            for event, elem in ET.iterparse(xml_file, events = ("end",)):
                if (elem.tag == drug_tag
                    and elem.get("type") == "small molecule"
                    and elem.find(primary_tag) is not None
                    and elem.find(approved_tag) is not None
                    and elem.find(withdrawn_tag) is None):   # ← exclude withdrawn
                    
                    out.write(f"\n{'='*60}\n")
                    out.write(f"DRUG {drugs_seen + 1}  |  type={elem.get('type')}\n")
                    out.write(f"DRUG: {elem.find(primary_tag).text}\n")
                    out.write(f"{'='*60}\n")
                    print_tree(elem, NS, out=out)

                    drugs_seen += 1
                    elem.clear()

                    if drugs_seen >= 10: 
                        break

Streaming: drugbank_full_database_V5.1.14.xml


NameError: name 'primary_tag' is not defined